# MLP Combine with vector Space: 

Each word is mapped to a point in a continuous vector space via an embedding lookup.
Embeddings from the context are concatenated and passed through an MLP, which transforms these coordinates into logits.
Softmax normalizes the logits into a probability distribution over the vocabulary

In [2]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt

c:\Users\Yashraj Sharma\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
words=open('names.txt','r').read().splitlines()
print(words[:8])

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [5]:
len(words)

32033

In [6]:
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [7]:
# building the dataset
block_size=3
X,Y=[],[]
for w in words[:5]:
    print(w)
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '----->', itos[ix])
        context=context[1:]+[ix]

X=torch.tensor(X)
Y=torch.tensor(Y)

emma
... -----> e
..e -----> m
.em -----> m
emm -----> a
mma -----> .
olivia
... -----> o
..o -----> l
.ol -----> i
oli -----> v
liv -----> i
ivi -----> a
via -----> .
ava
... -----> a
..a -----> v
.av -----> a
ava -----> .
isabella
... -----> i
..i -----> s
.is -----> a
isa -----> b
sab -----> e
abe -----> l
bel -----> l
ell -----> a
lla -----> .
sophia
... -----> s
..s -----> o
.so -----> p
sop -----> h
oph -----> i
phi -----> a
hia -----> .


In [8]:
print("Shape of X:",X.shape) # Means there are 3 characters in the context in 32 training examples.
print("Data type of X:",X.dtype)
print("Shape of Y:",Y.shape) # Means there is only 1 output character in 32 training examples.
print("Data type of Y:",Y.dtype)

Shape of X: torch.Size([32, 3])
Data type of X: torch.int64
Shape of Y: torch.Size([32])
Data type of Y: torch.int64


In [9]:
print(X[:10])

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22]])


### Input Embeddings:

In [10]:
### Generating the embeddings

C= torch.randn((27,2)) # 27 is the number of characters in the vocabulary and 2 is the size of the embedding vector for each character menas 2 dimensions for each character. We can choose any number of dimensions for the embedding vector, but 2 is a good choice for visualization purposes.
print(C[:10])

tensor([[-0.7670, -0.5105],
        [ 1.3901, -0.5839],
        [ 0.4593,  0.2269],
        [ 0.5976, -0.9737],
        [ 0.3430, -0.2544],
        [-0.1985,  0.4820],
        [ 0.3176, -0.5102],
        [ 0.9515,  0.6392],
        [-0.5075, -0.1789],
        [ 0.7029,  2.1662]])


In [11]:
emb = C[X] # (32,3,2) # 32 training examples, 3 characters in the context, and 2 dimensions for each character.(batch, context, embedding)
# means a context window of 3 characters is being mapped to a 2-dimensional embedding vector for each character in the context. So, we have a total of 32 training examples, and each example consists of a context window of 3 characters, which is represented as a 2-dimensional embedding vector for each character in the context.
# so in this, each particular charater in context is being mapped to a 2-dimensional embedding vector, and we have a total of 32 training examples, where each example consists of a context window of 3 characters, and each character in the context is represented as a 2-dimensional embedding vector.
print(emb[1][2][0]) # this is the first dimension of the embedding vector for the third character in the context of the second training example becuase of 0-based indexing.

tensor(-0.1985)


### Hidden layer

In [ ]:
W1 = torch.randn((6,100)) # 6 is the number of input features (3 characters in the context * 2 dimensions for each character) and 100 is the number of neurons in the hidden layer.
b1 = torch.randn(100) # 100 is the number of neurons in the hidden layer.

# But There is a problem here, usually we to emb @ W1 + b1, but the shape of emb is (32,3,2) and the shape of W1 is (6,100), so we need to reshape emb to (32,6) before we can perform the matrix multiplication.
#emb_reshaped = torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],dim=1) # this is the first, second and third character in the context for all 32 training examples, reshaped to (32,6)
# emb_reshaped = torch.cat(torch.unbind(emb,dim=1)) # this will give us a list of 3 tensors, each of shape (32,2), which is the first, second and third character in the context for all 32 training examples.
emb_reshaped = emb.view(emb.shape[0], -1) # this will reshape the emb tensor to (32,6) by concatenating the 3 characters in the context for all 32 training examples.
# Now we can perform the matrix multiplication and add the bias to get the output of the hidden layer. shape will be (32,6) @ (6,100) + (100,) = (32,100)

In [17]:
h= torch.tanh(emb_reshaped @ W1 + b1) # this will give us the output of the hidden layer, which is a tensor of shape (32,100), where 32 is the number of training examples and 100 is the number of neurons in the hidden layer. Each row in this tensor represents the output of the hidden layer for a particular training example.
h

tensor([[-0.7276, -0.5847, -0.9658,  ..., -0.1508, -0.7449, -0.4294],
        [-0.9989, -0.9301,  0.8790,  ..., -0.4518, -0.8395,  0.7256],
        [ 0.3141, -0.8912, -0.9428,  ..., -0.0443, -0.1738, -0.9746],
        ...,
        [ 0.9914, -0.9992,  0.1463,  ..., -0.1894,  0.9620, -0.9108],
        [-0.9968, -0.9260,  1.0000,  ...,  0.9993, -0.9983,  0.4797],
        [ 0.9817, -0.9725,  0.9773,  ..., -0.9916, -0.9661, -0.7940]])

### Output Layer:

In [ ]:
w2= torch.randn((100,27)) # 100 is the number of neurons in the hidden layer and 27 is the number of characters in the vocabulary, which is also the number of output classes.
b2= torch.randn(27) # 27 is the number of characters in the vocabulary, which is also the number of output classes.

logits = h @ w2 + b2 # this will give us the logits for each of the 27 output classes for each of the 32 training examples. The shape of logits will be (32,27), where 32 is the number of training examples and 27 is the number of output classes.
count= logits.exp() # this will give us the unnormalized probabilities for each of the 27 output classes for each of the 32 training examples. The shape of count will be (32,27), where 32 is the number of training examples and 27 is the number of output classes.
prob= count / count.sum(dim=1, keepdim=True) # this will give us the normalized probabilities for each of the 27 output classes for each of the 32 training examples. The shape of prob will be (32,27), where 32 is the number of training examples and 27 is the number of output classes. Each row in this tensor represents a probability distribution over the 27 output classes for a particular training example.
prob[torch.arange(32),Y] # this will give us the probabilities of the correct output class for each of the 32 training examples. The shape of this tensor will be (32,), where each element represents the probability of the correct output class for a particular training example. We can use these probabilities to compute the loss and perform backpropagation to update the weights and biases of the model during training.

tensor([3.4612e-11, 9.9996e-01, 9.9503e-01, 5.9273e-12, 1.1903e-07, 6.7779e-13,
        2.1450e-10, 1.7918e-13, 8.5824e-06, 1.4449e-10, 8.0128e-10, 4.7632e-09,
        6.5611e-17, 1.4753e-06, 3.5096e-05, 4.3023e-09, 8.5632e-16, 1.1790e-05,
        1.5369e-05, 3.8407e-05, 1.1495e-07, 1.8954e-06, 2.2932e-04, 1.0110e-11,
        7.7376e-10, 7.3691e-17, 4.3086e-11, 2.4767e-08, 1.5329e-12, 6.7482e-09,
        1.6274e-07, 3.8204e-08])

## Negative LogLikelihood:

In [24]:
loss=-prob[torch.arange(32),Y].log().mean() # this will give us the average negative log-likelihood loss for the 32 training examples. The shape of this tensor will be a scalar, which represents the average loss for the batch of training examples. We can use this loss to perform backpropagation and update the weights and biases of the model during training.
loss

tensor(19.0170)

The above way of computing the loss is not very efficient because we are computing the probabilities for all the output classes, which is not necessary. We can directly compute the loss from the logits without computing the probabilities, which is more efficient and numerically stable.

We can use the cross-entropy loss function, which combines the softmax function and the negative log-likelihood loss into a single function. The cross-entropy loss can be computed as follows:

loss = torch.nn.functional.cross_entropy(logits, Y) # scalar

why this is better?

reason 1: It is more efficient because we are not computing the probabilities for all the output classes, which is not necessary. We only need the logits for the correct output class to compute the loss, and the cross-entropy loss function allows us to do that directly without computing the probabilities.

reason 2: It is more numerically stable because it avoids the potential issues of computing the probabilities, which can lead to very small values that can cause underflow when taking the logarithm. The cross-entropy loss function is designed to handle this issue and provides a more stable way to compute the loss directly from the logits.

### ALL CODE IN CLEAR WAY:

### Weights and Bias:

In [25]:
G=torch.Generator().manual_seed(2147483647)
C= torch.randn((27,2), generator=G) 
W1 = torch.randn((6,100), generator=G)
b1 = torch.randn(100, generator=G)
W2 = torch.randn((100,27), generator=G)
b2 = torch.randn(27, generator=G)
parameters = [C, W1, b1, W2, b2]

### Gradient Calculation Turn on:

In [31]:
for p in parameters:
    p.requires_grad = True

In [32]:
for _ in range(1000):
    # Forward pass
    embeddings = C[X] # (32,3,2)
    embeddings_reshaped = embeddings.view(embeddings.shape[0], -1) # (32,6)
    hidden_layer_output = torch.tanh(embeddings_reshaped @ W1 + b1) # (32,100)
    logits = hidden_layer_output @ W2 + b2 # (32,27)
    loss=F.cross_entropy(logits, Y) # this will give us the average cross-entropy loss for the 32 training examples. The shape of this tensor will be a scalar, which represents the average loss for the batch of training examples. We can use this loss to perform backpropagation and update the weights and biases of the model during training
  
    # Backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update parameters using gradient descent
    learning_rate = 0.1
    with torch.no_grad():
        for p in parameters:
            p -= learning_rate * p.grad

print(loss.item())

0.25613829493522644
